# 04 · Meeting the Quaternion

### Recap & why now
Notebook 03 found the hole in Euler angles and Notebook 02 found the awkwardness in
composing rotations. A quaternion fixes both, at the cost of one extra number and a
few minutes of unfamiliarity.

The unfamiliarity is worth pushing through, because from Notebook 06 onward the
simulator's orientation state **is** a quaternion, and every controller in the project
computes its attitude error from one.

### Learning objectives
1. Read a quaternion as an **axis and an angle**, which is the useful interpretation.
2. Use the Hamilton product to **compose** rotations, and get the order right.
3. Rotate a vector two ways — matrix and sandwich product — and verify they agree.
4. Explain the **double cover**: why $q$ and $-q$ are the same orientation.
5. State the conventions this project fixes, and why each one matters.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · Four numbers, read geometrically

$$q = [\,q_w,\; q_x,\; q_y,\; q_z\,], \qquad q_w^2 + q_x^2 + q_y^2 + q_z^2 = 1$$

Euler's rotation theorem says any orientation is *some* rotation by an angle $\alpha$
about *some* axis $\hat{n}$. The quaternion just stores that pair:

$$q = \left[\cos\tfrac{\alpha}{2},\;\; \hat{n}\sin\tfrac{\alpha}{2}\right]$$

So $q_w$ is a "how much" and $[q_x, q_y, q_z]$ is a "which way". The halved angle is the
price of the convenience that follows. Identity — no rotation — is $[1, 0, 0, 0]$.

In [ ]:
print("  rotation                   quaternion              |q|")
for axis, deg, label in [([1, 0, 0], 0,   "no rotation      "),
                         ([1, 0, 0], 90,  "90° about x (roll)"),
                         ([0, 1, 0], 90,  "90° about y      "),
                         ([0, 0, 1], 180, "180° about z     "),
                         ([1, 1, 1], 120, "120° about [1,1,1]")]:
    q = axis_angle_to_quat(axis, np.deg2rad(deg))  # Build it from the geometric description.
    print("  %s %-24s %.6f" % (label, np.round(q, 4), np.linalg.norm(q)))

q90 = axis_angle_to_quat([1, 0, 0], np.pi/2)
print("\n90° about x is [cos45°, sin45°, 0, 0] = [%.4f, %.4f, 0, 0] ✔" % (np.cos(np.pi/4), np.sin(np.pi/4)))
print("Every one has length exactly 1 — the constraint that makes four numbers describe three")
print("degrees of freedom, and the reason there is no singularity to fall into.")

## 2 · Composing without ambiguity

Rotations compose by multiplication, exactly as matrices do:

$$q_1 \otimes q_2 \quad\text{means}\quad \text{apply } q_2 \text{ first, then } q_1$$

Same reading order as matrices, so the two never disagree. And since quaternions do not
commute either, Notebook 02's lesson survives intact — it is simply expressed in four
numbers instead of nine.

In [ ]:
q_roll  = axis_angle_to_quat([1, 0, 0], np.pi/2)   # 90° roll.
q_yaw   = axis_angle_to_quat([0, 0, 1], np.pi/2)   # 90° yaw.

print("two 90° rolls        :", np.round(quat_multiply(q_roll, q_roll), 4))
print("one 180° roll        :", np.round(axis_angle_to_quat([1, 0, 0], np.pi), 4), " <- the same ✔")
print("q (x) q* (undo it)   :", np.round(quat_multiply(q_roll, quat_conjugate(q_roll)), 12))

print("\nroll then yaw:", np.round(quat_multiply(q_yaw, q_roll), 4))
print("yaw then roll:", np.round(quat_multiply(q_roll, q_yaw), 4), " <- DIFFERENT, as it must be")

v = np.array([1.0, 0.0, 0.0])
a = quat_rotate(quat_multiply(q_yaw, q_roll), v)
b = quat_rotate(quat_multiply(q_roll, q_yaw), v)
print("and the two orders send the nose %.1f° apart." %
      np.degrees(np.arccos(np.clip(a @ b, -1, 1))))

## 3 · Rotating a vector, two ways

You can build the matrix and multiply, or sandwich the vector between $q$ and its
conjugate:

$$v_{\text{world}} = R(q)\, v_{\text{body}}
\qquad\text{or}\qquad
[0, v_{\text{world}}] = q \otimes [0, v_{\text{body}}] \otimes q^{*}$$

These must agree exactly. Checking that they do is the most valuable single test in this
notebook, because a wrong sign in `quat_to_rotmat` produces a simulator that looks
plausible and flies backwards.

In [ ]:
q = euler_to_quat(0.35, -0.20, 1.10)
v_body = np.array([0.4, -1.2, 2.0])

print("sandwich product vs matrix : %.2e" % np.abs(quat_rotate(q, v_body) - quat_to_rotmat(q) @ v_body).max())
print("R^T R - I                  : %.2e" % np.abs(quat_to_rotmat(q).T @ quat_to_rotmat(q) - np.eye(3)).max())
print("det R                      : %.12f" % np.linalg.det(quat_to_rotmat(q)))
print("matrix -> quaternion -> matrix: %.2e" %
      np.abs(quat_to_rotmat(quat_from_rotmat(quat_to_rotmat(q))) - quat_to_rotmat(q)).max())

T = 12.0                                           # A thrust magnitude, in newtons.
print("\n  attitude (roll, pitch, yaw)     thrust in world [N]     vertical share")
for r, p_, y_ in [(0, 0, 0), (30, 0, 0), (0, 30, 40)]:
    qq = euler_to_quat(*np.deg2rad([r, p_, y_]))
    F = quat_rotate(qq, [0.0, 0.0, T])             # The body thrust, seen from the world.
    print("  %4d %6d %6d          %-24s %5.0f%%" % (r, p_, y_, np.round(F, 2), 100*F[2]/T))

## 4 · The double cover, and why a controller cares

Negate all four components and you get the same rotation matrix. Every orientation has
**two** quaternion representations, $q$ and $-q$ — a fact with a practical consequence.

An attitude controller computes an error quaternion and turns its vector part into a
command. If the error comes back with a negative scalar part, the rotation it describes
is the *long way round*: 350° instead of 10°. One line of code fixes it, and Notebook 08
contains that line.

In [ ]:
q_a = euler_to_quat(0.4, 0.2, -0.7)
print("max |R(q) - R(-q)| = %.2e  -> identical rotations" %
      np.abs(quat_to_rotmat(q_a) - quat_to_rotmat(-q_a)).max())

def rotation_angle(q):
    """The angle this quaternion rotates through, in degrees."""
    return np.degrees(2*np.arccos(np.clip(abs(q[0]), -1, 1)))

print("\n  error quaternion              scalar part   rotation it describes")
for label, q_e in [("small error       ", axis_angle_to_quat([0, 0, 1], np.deg2rad(10))),
                   ("the same, negated ", -axis_angle_to_quat([0, 0, 1], np.deg2rad(10)))]:
    short = 2*np.degrees(np.arcsin(np.clip(np.linalg.norm(q_e[1:]), 0, 1)))
    long_way = 360 - short
    print("  %s %-22s %+8.4f %10.1f°" %
          (label, np.round(q_e, 4), q_e[0], short if q_e[0] >= 0 else long_way))

print("\nThe fix, in full:  if q_e[0] < 0: q_e = -q_e")
print("Leave it out and a drone with a tiny heading error will spin almost all the way round")
print("to correct it. That is not a wobble, it is a flip.")

## 🧪 Try it yourself

**E1.** A quaternion has four numbers for three degrees of freedom. What does the extra
number buy, and what would break if you dropped the unit-length constraint?

**E2.** Write `angle_between(q1, q2)` giving the smallest rotation taking one attitude
to the other, and check it on a few pairs you can verify by hand.

In [ ]:
# --- Solution E1 ---
print("E1: it buys the absence of a singularity. Notebook 03 showed that three numbers cannot")
print("    cover the space of rotations smoothly — some attitude always loses a degree of freedom.")
print("    Four numbers with one constraint can. Drop the constraint and the quaternion stops")
print("    being a rotation: it starts scaling vectors as well as turning them.")
q_bad = np.array([1.0, 0.0, 0.0, 0.0])*1.05        # A quaternion 5% too long.
v = np.array([0.0, 0.0, 1.0])
print("    |q| = %.2f rotates [0,0,1] to %s — length %.4f instead of 1." %
      (np.linalg.norm(q_bad),
       np.round(quat_multiply(quat_multiply(q_bad, np.array([0., *v])), quat_conjugate(q_bad))[1:], 4),
       np.linalg.norm(quat_multiply(quat_multiply(q_bad, np.array([0., *v])), quat_conjugate(q_bad))[1:])))
print("    A drone whose orientation is 5%% too long silently gains 10%% of thrust. Notebook 05")
print("    is about keeping that from happening.")

# --- Solution E2 ---
def angle_between(q1, q2):
    """Smallest rotation angle taking attitude q1 to attitude q2, in degrees."""
    q_e = quat_multiply(quat_conjugate(quat_normalize(q1)), quat_normalize(q2))
    if q_e[0] < 0:
        q_e = -q_e                                 # Short way round — the same fix as Section 4.
    return np.degrees(2*np.arccos(np.clip(q_e[0], -1, 1)))

print("\nE2:  pair                                      angle")
for label, a_, b_ in [("identical                        ", euler_to_quat(0.2, 0, 0), euler_to_quat(0.2, 0, 0)),
                      ("10° apart in roll                ", euler_to_quat(0, 0, 0), euler_to_quat(np.deg2rad(10), 0, 0)),
                      ("90° apart in yaw                 ", euler_to_quat(0, 0, 0), euler_to_quat(0, 0, np.pi/2)),
                      ("170° vs -170° yaw (near the wrap)", euler_to_quat(0, 0, np.deg2rad(170)),
                                                            euler_to_quat(0, 0, np.deg2rad(-170)))]:
    print("    %s %8.2f°" % (label, angle_between(a_, b_)))
print("    The last row is the one that matters: 170° and -170° are 20° apart, not 340°, and the")
print("    sign fix is what produces that answer.")

## 🚁 Mini-project: one axis, one angle

Animate a rotation the way a quaternion describes it — a single steady turn about one
fixed axis — and watch the drone take the direct route between two attitudes. There is
no roll-then-pitch-then-yaw sequence here; there never was one.

In [ ]:
q_start = euler_to_quat(0, 0, 0)
q_end   = euler_to_quat(np.deg2rad(35), np.deg2rad(-25), np.deg2rad(70))
q_delta = quat_multiply(quat_conjugate(q_start), q_end)
if q_delta[0] < 0:
    q_delta = -q_delta                             # Take the short way, as always.
axis = q_delta[1:]/max(np.linalg.norm(q_delta[1:]), 1e-9)
total = 2*np.arccos(np.clip(q_delta[0], -1, 1))
print("that attitude change is a single %.1f° turn about the axis %s" % (np.degrees(total), np.round(axis, 3)))

n = 80
poses = [quat_multiply(q_start, axis_angle_to_quat(axis, total*k/(n-1))) for k in range(n)]
poses += poses[::-1]                               # Sweep back, so the clip loops.

fig = plt.figure(figsize=(6.2, 5.2))
ax = fig.add_subplot(111, projection="3d")

def frame(k):
    ax.clear()
    draw_quad(ax, [0, 0, 0], poses[k], scale=2.2)
    ax.quiver(0, 0, 0, *(axis*1.1), color="C4", lw=2, arrow_length_ratio=0.15)   # The rotation axis.
    set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-1.0, 1.2))
    rpy = np.degrees(quat_to_euler(poses[k]))
    ax.set_title("one axis, one angle   |   roll %5.1f°  pitch %5.1f°  yaw %5.1f°" % tuple(rpy),
                 fontsize=9)
    ax.view_init(elev=20, azim=-60)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(poses), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **📐 Quaternion conventions fixed for this project.**
> Order $[q_w, q_x, q_y, q_z]$, scalar first — SciPy puts the scalar *last*, which is a
> classic silent bug. Hamilton multiplication, with $q_1 \otimes q_2$ meaning "apply
> $q_2$ first". The quaternion rotates **body → world**. Angular velocity $\omega$ is in
> the **body** frame, which is what a gyroscope measures. Renormalise after every
> integration step.

> **🤖 Robotics connection.** Every flight controller carries orientation as a
> quaternion and converts to Euler angles only for the telemetry screen. The reason is
> Notebook 03's numbers: an aerobatic drone passing through vertical would meet the
> Euler singularity several times a flight, and a filter that divides by $\cos\theta$
> there fails in a single time step.

**Where next.** Notebook 05 makes the quaternion move: given a gyroscope reading, how
does the orientation evolve, and why must it be renormalised?